# MOSAIC Phase 1: Data Audit, EDA, and Condition Selection

Multi condition facial skin severity prediction. This notebook is the first step of the
project and it has one job: find out what the dataset actually contains, before any model
code is written.

**Disclaimer.** MOSAIC is an academic screening aid. It is not a medical device and it does
not diagnose disease.

## What this notebook does

1. Discovers the attached dataset structure without assuming any layout.
2. Verifies the label schema against what the project proposal assumes, and reports every
   mismatch instead of working around it.
3. Audits image integrity, resolution, and color mode.
4. Measures label distribution, missingness, and imbalance per condition.
5. Measures redundancy between conditions with Spearman correlation.
6. Detects near duplicate images with perceptual hashing and assigns group ids, so the
   Phase 2 split can keep duplicates inside one split and avoid leakage.
7. Checks whether faces are already cropped and centered, and recommends whether a face
   detector is needed.
8. Produces a clean manifest that records which rows are usable and why.
9. Applies the condition selection rule and proposes the final condition set.

## What this notebook deliberately does not do

1. It does not train anything. There is no model here, so no GPU is required.
2. It does not modify or delete any source data. The dataset mount is read only.
3. It does not impute missing labels. A missing label stays missing.
4. It does not finalize the condition list. That decision needs human approval at Gate 1.

## How to run this on Kaggle

1. Create a new Kaggle notebook.
2. Settings, then Accelerator, then **None**. This notebook is CPU only, so running it on a
   GPU would waste weekly GPU quota for no benefit.
3. Settings, then Internet, then **On**. The install cell needs it.
4. Add Data, then attach `killa92/facial-skin-analysis-and-type-classification`.
5. Run every cell from top to bottom.
6. Download the contents of `/kaggle/working/` when the run finishes, then report the Gate 1
   summary printed by the last cell.

## 1. Setup

The install cell adds only what the Kaggle base image does not already provide. Everything
else in this notebook comes from the preinstalled scientific Python stack.

In [ ]:
!pip install -q imagehash

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import sys
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Literal

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageFile
from scipy import stats
from tqdm.auto import tqdm

# Truncated files should raise instead of silently decoding to garbage, because a silently
# broken image would become a silently wrong training sample.
ImageFile.LOAD_TRUNCATED_IMAGES = False

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
sns.set_theme(style="whitegrid", context="notebook")

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("opencv :", cv2.__version__)

### 1.1 Configuration

Every threshold used later in this notebook is declared here, so no later cell contains a
magic number. The selection thresholds come from the project requirements document, section
7.1. The values under `Assumptions to verify` are what the written proposal claims about the
dataset. They are treated as claims to be tested, never as facts.

In [ ]:
@dataclass
class AuditConfig:
    """All paths and thresholds used by this notebook.

    Attributes:
        input_root: Read only Kaggle mount holding every attached dataset.
        working_root: Writable Kaggle directory. Everything produced here is downloadable.
        dataset_name_hints: Substrings used to pick the facial skin dataset when several
            datasets are attached.
        seed: Seed for every sampling operation, so reruns are identical.
        image_extensions: File suffixes treated as images.
        table_extensions: File suffixes treated as candidate annotation tables.
        deep_image_check: If True, fully decode every image to detect corruption. Slower but
            stricter than a header only check.
        min_image_side: Images with any side below this are flagged as too small.
        phash_size: Perceptual hash size. 8 produces a 64 bit hash.
        phash_max_distance: Hamming distance at or below which two images share a group id.
        max_missing_rate: Selection rule 1. A condition needs a missing rate below this.
        min_levels_with_support: Selection rule 2. Minimum number of levels that must each
            reach the support threshold.
        min_train_samples_per_level: Selection rule 2. Required samples per level, counted in
            the future training split.
        assumed_train_fraction: Used only to convert the training split requirement above
            into a whole dataset requirement, since the split does not exist yet.
        min_normalized_entropy: Selection rule 3. Minimum normalized label entropy.
        redundancy_spearman: Above this absolute correlation, two conditions are treated as
            redundant and the lower ranked one is dropped.
        forced_conditions: Conditions that must appear in the final set regardless of rank.
        target_condition_count: How many conditions the project plans to model.
        expected_level_min: Proposal assumption about the lowest severity score.
        expected_level_max: Proposal assumption about the highest severity score.
        expected_condition_count: Proposal assumption about the number of scored conditions.
        max_unique_for_ordinal: A numeric column with at most this many distinct values is
            treated as a candidate ordinal severity column.
        face_sample_size: Number of images sampled for the face framing check.
        show_sample_images: Whether to render image grids. Set to False to keep the saved
            notebook free of dataset imagery.
    """

    input_root: Path = Path("/kaggle/input")
    working_root: Path = Path("/kaggle/working")
    dataset_name_hints: tuple[str, ...] = ("facial-skin", "facial_skin", "skin-analysis")

    seed: int = 42
    image_extensions: tuple[str, ...] = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    table_extensions: tuple[str, ...] = (".csv", ".xlsx", ".xls", ".json", ".parquet", ".tsv")

    deep_image_check: bool = False
    min_image_side: int = 100

    phash_size: int = 8
    phash_max_distance: int = 8

    max_missing_rate: float = 0.20
    min_levels_with_support: int = 3
    min_train_samples_per_level: int = 30
    assumed_train_fraction: float = 0.70
    min_normalized_entropy: float = 0.50
    redundancy_spearman: float = 0.80
    forced_conditions: tuple[str, ...] = ("acne",)
    target_condition_count: int = 6

    expected_level_min: int = 0
    expected_level_max: int = 5
    expected_condition_count: int = 18
    max_unique_for_ordinal: int = 12

    face_sample_size: int = 50
    show_sample_images: bool = True

    @property
    def figures_dir(self) -> Path:
        return self.working_root / "figures"

    @property
    def tables_dir(self) -> Path:
        return self.working_root / "tables"

    @property
    def min_overall_samples_per_level(self) -> int:
        """Whole dataset support needed so the future train split reaches its requirement."""
        return math.ceil(self.min_train_samples_per_level / self.assumed_train_fraction)


CFG = AuditConfig()
CFG.figures_dir.mkdir(parents=True, exist_ok=True)
CFG.tables_dir.mkdir(parents=True, exist_ok=True)

print("Resolved configuration")
for key, value in asdict(CFG).items():
    print(f"  {key:32s} {value}")
print(f"  {'min_overall_samples_per_level':32s} {CFG.min_overall_samples_per_level}")

### 1.2 Reproducibility and helpers

Seeding matters even in an audit, because this notebook samples images for the framing check
and for the example grids. Without a fixed seed, two runs would show different examples and
the written findings would not match the figures.

In [ ]:
def seed_everything(seed: int) -> None:
    """Seed every random source used in this notebook.

    Args:
        seed: Seed value applied to Python, NumPy, and the hash seed.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def log(message: str) -> None:
    """Print a timestamped progress message.

    Args:
        message: Text to print.
    """
    print(f"[{time.strftime('%H:%M:%S')}] {message}")


def save_fig(fig: plt.Figure, name: str) -> None:
    """Save a figure as 300 dpi PNG and as PDF.

    Args:
        fig: Matplotlib figure to save.
        name: File stem, for example "fig01_level_distribution".
    """
    for suffix in ("png", "pdf"):
        fig.savefig(CFG.figures_dir / f"{name}.{suffix}", dpi=300, bbox_inches="tight")
    log(f"saved figure {name}")


def save_table(frame: pd.DataFrame, name: str) -> None:
    """Save a dataframe as CSV in the working tables directory.

    Args:
        frame: Dataframe to save.
        name: File stem without extension.
    """
    path = CFG.tables_dir / f"{name}.csv"
    frame.to_csv(path, index=False)
    log(f"saved table {name} with shape {frame.shape}")


seed_everything(CFG.seed)
AUDIT: dict[str, Any] = {"config": {k: str(v) for k, v in asdict(CFG).items()}}
log("setup complete")

## 2. Dataset discovery

Nothing below assumes a directory layout. The notebook walks the Kaggle mount, counts what it
finds, and reports it. If the dataset is laid out differently from what the proposal expects,
that shows up here rather than as a confusing failure ten cells later.

In [ ]:
def summarize_tree(root: Path, max_depth: int = 3) -> pd.DataFrame:
    """Summarize a directory tree by counting files per directory.

    Args:
        root: Directory to walk.
        max_depth: Maximum depth below root to report individually.

    Returns:
        One row per directory with file counts and the extensions present.
    """
    rows: list[dict[str, Any]] = []
    for current, _dirs, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.relative_to(root).parts)
        if depth > max_depth:
            continue
        extensions = Counter(Path(f).suffix.lower() for f in files)
        rows.append(
            {
                "directory": str(current_path.relative_to(root)) or ".",
                "depth": depth,
                "n_files": len(files),
                "extensions": ", ".join(f"{ext or 'none'}:{n}" for ext, n in extensions.most_common(6)),
            }
        )
    return pd.DataFrame(rows).sort_values(["depth", "directory"]).reset_index(drop=True)


if not CFG.input_root.exists():
    raise FileNotFoundError(
        f"{CFG.input_root} does not exist. This notebook is meant to run on Kaggle with at "
        f"least one dataset attached through Add Data."
    )

attached = sorted(p for p in CFG.input_root.iterdir() if p.is_dir())
print(f"Datasets attached to this session: {len(attached)}")
for path in attached:
    print(f"  {path.name}")

if not attached:
    raise FileNotFoundError(
        "No dataset is attached. Use Add Data and attach "
        "killa92/facial-skin-analysis-and-type-classification, then rerun."
    )

tree = summarize_tree(CFG.input_root, max_depth=3)
display(tree.head(60))

In [ ]:
def resolve_dataset_root(config: AuditConfig, available: list[Path]) -> Path:
    """Pick the directory that holds the main facial skin dataset.

    Args:
        config: Audit configuration holding the name hints.
        available: Directories currently attached under the Kaggle input mount.

    Returns:
        The chosen dataset root.

    Raises:
        FileNotFoundError: If no directory matches and the choice is ambiguous.
    """
    matches = [p for p in available if any(h in p.name.lower() for h in config.dataset_name_hints)]
    if len(matches) == 1:
        return matches[0]
    if not matches and len(available) == 1:
        return available[0]
    if len(matches) > 1:
        raise FileNotFoundError(
            f"Several attached datasets match the name hints {config.dataset_name_hints}: "
            f"{[p.name for p in matches]}. Set DATASET_ROOT manually in the next cell."
        )
    raise FileNotFoundError(
        f"No attached dataset matches {config.dataset_name_hints}. "
        f"Attached: {[p.name for p in available]}. Set DATASET_ROOT manually."
    )


DATASET_ROOT = resolve_dataset_root(CFG, attached)
print("Dataset root:", DATASET_ROOT)
AUDIT["dataset_root"] = str(DATASET_ROOT)

In [ ]:
def collect_files(root: Path, suffixes: Iterable[str]) -> list[Path]:
    """List every file under a root whose suffix is in the given set.

    Args:
        root: Directory to search recursively.
        suffixes: Lowercase suffixes to keep, including the leading dot.

    Returns:
        Sorted list of matching paths.
    """
    wanted = {s.lower() for s in suffixes}
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in wanted)


image_files = collect_files(DATASET_ROOT, CFG.image_extensions)
table_files = collect_files(DATASET_ROOT, CFG.table_extensions)

print(f"Image files found : {len(image_files):,}")
print(f"Table files found : {len(table_files)}")
print()

if table_files:
    table_summary = pd.DataFrame(
        {
            "file": [str(p.relative_to(DATASET_ROOT)) for p in table_files],
            "size_mb": [round(p.stat().st_size / 1e6, 3) for p in table_files],
        }
    ).sort_values("size_mb", ascending=False)
    display(table_summary)
else:
    print("No tabular annotation file was found under the dataset root.")

if image_files:
    print("\nExample image paths:")
    for path in image_files[:5]:
        print("  ", path.relative_to(DATASET_ROOT))

LAYOUT: Literal["tabular", "folder_per_class", "unknown"]
if table_files and image_files:
    LAYOUT = "tabular"
elif image_files and not table_files:
    LAYOUT = "folder_per_class"
else:
    LAYOUT = "unknown"

print(f"\nInferred layout: {LAYOUT}")
AUDIT["layout"] = LAYOUT
AUDIT["n_image_files"] = len(image_files)
AUDIT["n_table_files"] = len(table_files)

## 3. Label schema verification

This is the gate that protects the rest of the project. The proposal assumes integer severity
scores from 0 to 5 across 18 conditions. If that is wrong, then the head count, the loss
choice, and the metric definitions all change, and finding out now costs nothing while finding
out in Phase 5 costs a week.

In [ ]:
def load_tabular(path: Path) -> pd.DataFrame:
    """Load a tabular annotation file, inferring the format from its suffix.

    Args:
        path: File to load.

    Returns:
        The loaded dataframe.

    Raises:
        ValueError: If the suffix is not a supported tabular format.
    """
    suffix = path.suffix.lower()
    if suffix in {".csv", ".tsv", ".txt"}:
        separator = "\t" if suffix == ".tsv" else None
        return pd.read_csv(path, sep=separator, engine="python")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            with path.open("r", encoding="utf-8") as handle:
                payload = json.load(handle)
            return pd.json_normalize(payload)
    raise ValueError(f"Unsupported table format: {path.suffix}")


def build_frame_from_folders(files: list[Path], root: Path) -> pd.DataFrame:
    """Fall back to deriving labels from the parent folder name of each image.

    This path is used when the dataset ships no annotation table. It produces a single
    categorical label column, which is a very different schema from the ordinal severity
    scores the proposal assumes, and the verification table below will say so.

    Args:
        files: Image paths.
        root: Dataset root, used to build relative paths.

    Returns:
        A dataframe with an identifier, a relative path, and a folder label.
    """
    return pd.DataFrame(
        {
            "image_id": [p.stem for p in files],
            "image_path": [str(p.relative_to(root)) for p in files],
            "folder_label": [p.parent.name for p in files],
        }
    )


PRIMARY_TABLE_OVERRIDE: Path | None = None

if LAYOUT == "tabular":
    primary_table = PRIMARY_TABLE_OVERRIDE or max(table_files, key=lambda p: p.stat().st_size)
    log(f"loading primary annotation table: {primary_table.relative_to(DATASET_ROOT)}")
    raw = load_tabular(primary_table)
    AUDIT["primary_table"] = str(primary_table.relative_to(DATASET_ROOT))
elif LAYOUT == "folder_per_class":
    log("no annotation table present, deriving a frame from the folder structure instead")
    raw = build_frame_from_folders(image_files, DATASET_ROOT)
    AUDIT["primary_table"] = None
else:
    raise FileNotFoundError(
        "Neither images nor annotation tables were found under the dataset root. Check that "
        "the correct dataset is attached."
    )

print(f"shape: {raw.shape}")
print()
display(raw.head(10))
print()
display(raw.dtypes.to_frame("dtype"))

In [ ]:
def classify_columns(frame: pd.DataFrame, config: AuditConfig) -> pd.DataFrame:
    """Infer the role of every column so condition columns can be found automatically.

    Roles are heuristics, not ground truth. The point is to produce a table a human can read
    and correct, rather than to hard code column names the notebook has never seen.

    Args:
        frame: Loaded annotation frame.
        config: Audit configuration.

    Returns:
        One row per column with dtype, cardinality, missingness, sample values, and a role.
    """
    rows: list[dict[str, Any]] = []
    n_rows = len(frame)
    for column in frame.columns:
        series = frame[column]
        non_null = series.dropna()
        n_unique = int(non_null.nunique())
        sample = list(pd.unique(non_null)[:8])
        is_numeric = pd.api.types.is_numeric_dtype(series)
        looks_integral = bool(is_numeric and non_null.size and float(non_null.mod(1).abs().max()) == 0.0)
        name = str(column).lower()

        name_suggests_path = any(token in name for token in ("path", "file", "image", "img", "name"))
        values_suggest_path = bool(
            not is_numeric
            and non_null.size
            and any(str(v).lower().endswith(config.image_extensions) for v in sample)
        )

        # The image path test runs before the identifier test on purpose. A file name column is
        # usually unique as well, so checking uniqueness first would label it an identifier and
        # the notebook would then fail to find any image.
        if (name_suggests_path or values_suggest_path) and not is_numeric:
            role = "image_path"
        elif n_unique == n_rows and n_rows > 0:
            role = "identifier"
        elif is_numeric and looks_integral and 1 < n_unique <= config.max_unique_for_ordinal:
            role = "ordinal_candidate"
        elif is_numeric:
            role = "numeric_other"
        elif n_unique <= 30:
            role = "categorical_meta"
        else:
            role = "text"

        rows.append(
            {
                "column": column,
                "dtype": str(series.dtype),
                "role": role,
                "n_unique": n_unique,
                "n_missing": int(series.isna().sum()),
                "missing_rate": round(float(series.isna().mean()), 4),
                "min": float(non_null.min()) if is_numeric and non_null.size else None,
                "max": float(non_null.max()) if is_numeric and non_null.size else None,
                "sample_values": ", ".join(map(str, sample)),
            }
        )
    return pd.DataFrame(rows)


column_profile = classify_columns(raw, CFG)
display(column_profile)
save_table(column_profile, "phase1_column_profile")

In [ ]:
CONDITION_COLUMNS: list[str] = column_profile.loc[
    column_profile["role"] == "ordinal_candidate", "column"
].tolist()

ID_COLUMNS: list[str] = column_profile.loc[
    column_profile["role"] == "identifier", "column"
].tolist()

PATH_COLUMNS: list[str] = column_profile.loc[
    column_profile["role"] == "image_path", "column"
].tolist()

META_COLUMNS: list[str] = column_profile.loc[
    column_profile["role"].isin(["categorical_meta", "numeric_other", "text"]), "column"
].tolist()

print(f"Candidate condition columns ({len(CONDITION_COLUMNS)}):")
for column in CONDITION_COLUMNS:
    print("  ", column)
print(f"\nIdentifier columns: {ID_COLUMNS}")
print(f"Image path columns: {PATH_COLUMNS}")
print(f"Other metadata columns: {META_COLUMNS}")

In [ ]:
def observed_level_set(frame: pd.DataFrame, columns: list[str]) -> list[Any]:
    """Collect every distinct severity value present across the condition columns.

    Sorting is done on a type aware key, because a dataset that mixes numeric and string
    labels would otherwise raise a comparison error here rather than reporting the mix.

    Args:
        frame: Annotation frame.
        columns: Condition column names.

    Returns:
        Sorted list of distinct non null values, numbers first.
    """
    values: set[Any] = set()
    for column in columns:
        values.update(pd.unique(frame[column].dropna()).tolist())

    def sort_key(value: Any) -> tuple[int, float, str]:
        if isinstance(value, (int, float, np.integer, np.floating)):
            return (0, float(value), "")
        return (1, 0.0, str(value))

    return sorted(values, key=sort_key)


def find_sentinel_values(frame: pd.DataFrame, columns: list[str]) -> dict[str, int]:
    """Count values that commonly encode missingness in place of a real NaN.

    Args:
        frame: Annotation frame.
        columns: Condition column names.

    Returns:
        Mapping from sentinel value to the number of times it occurs.
    """
    candidates = [-1, 99, 999, -99, -999]
    counts: dict[str, int] = {}
    for candidate in candidates:
        total = int(sum(int((frame[c] == candidate).sum()) for c in columns))
        if total:
            counts[str(candidate)] = total
    return counts


def verify_schema(frame: pd.DataFrame, columns: list[str], config: AuditConfig) -> pd.DataFrame:
    """Compare the observed schema against the assumptions written in the proposal.

    Args:
        frame: Annotation frame.
        columns: Candidate condition columns.
        config: Audit configuration holding the assumptions.

    Returns:
        One row per assumption with observed value and a PASS, MISMATCH, or UNKNOWN status.
    """
    levels = observed_level_set(frame, columns) if columns else []
    numeric_levels = [v for v in levels if isinstance(v, (int, float, np.integer, np.floating))]
    sentinels = find_sentinel_values(frame, columns) if columns else {}
    has_acne = any("acne" in str(c).lower() for c in columns)
    subject_like = [c for c in frame.columns if any(t in str(c).lower() for t in ("subject", "person", "patient", "user"))]
    skintype_like = [c for c in frame.columns if "type" in str(c).lower()]

    def status(condition: bool) -> str:
        return "PASS" if condition else "MISMATCH"

    checks: list[dict[str, Any]] = [
        {
            "assumption": "An annotation table with per condition scores exists",
            "expected": "yes",
            "observed": "yes" if columns else "no ordinal candidate columns found",
            "status": status(bool(columns)),
        },
        {
            "assumption": "Number of scored conditions",
            "expected": config.expected_condition_count,
            "observed": len(columns),
            "status": status(len(columns) == config.expected_condition_count),
        },
        {
            "assumption": "Score range",
            "expected": f"{config.expected_level_min} to {config.expected_level_max}",
            "observed": f"{min(numeric_levels)} to {max(numeric_levels)}" if numeric_levels else "none",
            "status": status(
                bool(numeric_levels)
                and min(numeric_levels) >= config.expected_level_min
                and max(numeric_levels) <= config.expected_level_max
            ),
        },
        {
            "assumption": "Scores are integers",
            "expected": "yes",
            "observed": (
                "yes" if numeric_levels and all(float(v).is_integer() for v in numeric_levels)
                else "no numeric levels found" if not numeric_levels
                else "no, fractional values present"
            ),
            "status": status(bool(numeric_levels) and all(float(v).is_integer() for v in numeric_levels)),
        },
        {
            "assumption": "Missing labels encoded as NaN rather than a sentinel number",
            "expected": "NaN only",
            "observed": f"sentinels found: {sentinels}" if sentinels else "NaN only",
            "status": status(not sentinels),
        },
        {
            "assumption": "An acne column exists, required by the baseline and external eval",
            "expected": "yes",
            "observed": "yes" if has_acne else "no column name contains 'acne'",
            "status": status(has_acne),
        },
        {
            "assumption": "A subject or person id exists, used to prevent split leakage",
            "expected": "present or absent, both acceptable",
            "observed": subject_like or "none found",
            "status": "PASS" if subject_like else "UNKNOWN",
        },
        {
            "assumption": "A skin type column exists",
            "expected": "present or absent, both acceptable",
            "observed": skintype_like or "none found",
            "status": "PASS" if skintype_like else "UNKNOWN",
        },
    ]
    return pd.DataFrame(checks)


schema_check = verify_schema(raw, CONDITION_COLUMNS, CFG)
display(schema_check)
save_table(schema_check, "phase1_schema_verification")

LEVELS = observed_level_set(raw, CONDITION_COLUMNS) if CONDITION_COLUMNS else []
print(f"\nObserved severity levels across all condition columns: {LEVELS}")

n_mismatch = int((schema_check["status"] == "MISMATCH").sum())
AUDIT["schema_checks"] = schema_check.to_dict(orient="records")
AUDIT["observed_levels"] = [str(v) for v in LEVELS]
AUDIT["n_schema_mismatches"] = n_mismatch

if n_mismatch:
    print(
        f"\nATTENTION: {n_mismatch} assumption(s) from the proposal do not hold. "
        f"The audit continues so the full picture is visible, but the mismatches above must be "
        f"resolved with the team at Gate 1 before any modelling decision is locked."
    )
else:
    print("\nAll proposal assumptions about the label schema hold.")

## 4. Image integrity audit

Every annotation row is matched to a file on disk. Rows whose image is missing or unreadable
cannot be used for training, and counting them now prevents a silent shrink of the dataset
later.

In [ ]:
def build_basename_index(files: list[Path]) -> dict[str, list[Path]]:
    """Index image files by file name and by stem, for flexible path matching.

    Args:
        files: All image paths under the dataset root.

    Returns:
        Mapping from both the full file name and the bare stem to matching paths.
    """
    index: dict[str, list[Path]] = defaultdict(list)
    for path in files:
        index[path.name].append(path)
        index[path.stem].append(path)
    return index


def resolve_image_paths(
    frame: pd.DataFrame,
    path_column: str | None,
    root: Path,
    index: dict[str, list[Path]],
) -> pd.Series:
    """Map each annotation row to an absolute image path.

    Three strategies are tried in order: the value as a path relative to the dataset root, the
    value as a bare file name, and the value as a stem without extension.

    Args:
        frame: Annotation frame.
        path_column: Column holding a path or file name, or None if there is none.
        root: Dataset root.
        index: Output of build_basename_index.

    Returns:
        Series of resolved paths as strings, with None where no file matched.
    """
    if path_column is None:
        return pd.Series([None] * len(frame), index=frame.index, dtype=object)

    resolved: list[str | None] = []
    for value in frame[path_column].astype(str):
        direct = root / value
        if direct.is_file():
            resolved.append(str(direct))
            continue
        candidates = index.get(Path(value).name) or index.get(Path(value).stem)
        resolved.append(str(candidates[0]) if candidates else None)
    return pd.Series(resolved, index=frame.index, dtype=object)


def choose_path_column(
    frame: pd.DataFrame,
    candidates: list[str],
    root: Path,
    index: dict[str, list[Path]],
    probe_rows: int = 200,
) -> tuple[str | None, pd.DataFrame]:
    """Pick the column that actually resolves to image files, by testing each candidate.

    Guessing from the column name alone is fragile on a dataset nobody has inspected yet, so
    each candidate is scored on a sample of rows and the best one wins. The score table is
    returned so the choice is visible rather than hidden.

    Args:
        frame: Annotation frame.
        candidates: Columns that could plausibly point at an image.
        root: Dataset root.
        index: Output of build_basename_index.
        probe_rows: Number of rows to test per candidate.

    Returns:
        The winning column name or None, together with the score table.
    """
    probe = frame.head(probe_rows)
    scores: list[dict[str, Any]] = []
    for column in candidates:
        resolved = resolve_image_paths(probe, column, root, index)
        scores.append(
            {
                "column": column,
                "resolution_rate": round(float(resolved.notna().mean()), 4),
                "example": next((v for v in resolved if v is not None), None),
            }
        )
    score_frame = pd.DataFrame(scores).sort_values("resolution_rate", ascending=False)
    if score_frame.empty or float(score_frame.iloc[0]["resolution_rate"]) == 0.0:
        return None, score_frame
    return str(score_frame.iloc[0]["column"]), score_frame


basename_index = build_basename_index(image_files)

# Any text like column is a candidate, because the column that holds the file name is not
# always named in an obvious way.
path_candidates = list(dict.fromkeys(PATH_COLUMNS + ID_COLUMNS + [
    c for c in raw.columns if not pd.api.types.is_numeric_dtype(raw[c])
]))
path_column, path_scores = choose_path_column(raw, path_candidates, DATASET_ROOT, basename_index)
display(path_scores)
print("Column used to locate images:", path_column)

if path_column is None:
    raise RuntimeError(
        f"No column resolves to an image file. Columns tested: {path_candidates}. "
        f"Inspect the column profile and the example image paths printed in section 2, then set "
        f"path_column manually in this cell."
    )

data = raw.copy()
data["resolved_path"] = resolve_image_paths(data, path_column, DATASET_ROOT, basename_index)

n_resolved = int(data["resolved_path"].notna().sum())
print(f"Rows: {len(data):,}")
print(f"Rows with a resolvable image: {n_resolved:,} ({n_resolved / max(len(data), 1):.1%})")

if n_resolved == 0:
    raise RuntimeError(
        f"No annotation row could be matched to an image file. The column '{path_column}' was "
        f"used for matching. Inspect the column profile above and set path_column manually."
    )

ambiguous = sum(1 for paths in basename_index.values() if len(paths) > 1)
print(f"File names that map to more than one file on disk: {ambiguous}")
AUDIT["n_rows"] = len(data)
AUDIT["n_rows_with_image"] = n_resolved

In [ ]:
def read_image_meta(path: str, deep: bool) -> dict[str, Any]:
    """Read size, mode, and corruption status for one image.

    Args:
        path: Absolute image path.
        deep: If True, fully decode the pixels. Slower, but catches truncated files that a
            header only check would accept.

    Returns:
        Dictionary of metadata including a corrupted flag and an error message when relevant.
    """
    record: dict[str, Any] = {
        "width": None,
        "height": None,
        "mode": None,
        "size_kb": None,
        "corrupted": True,
        "error": None,
    }
    try:
        file_path = Path(path)
        record["size_kb"] = round(file_path.stat().st_size / 1024, 2)
        with Image.open(file_path) as image:
            record["width"], record["height"] = image.size
            record["mode"] = image.mode
            if deep:
                image.load()
            else:
                image.verify()
        record["corrupted"] = False
    except Exception as exc:  # noqa: BLE001 - any failure here means the file is unusable
        record["error"] = str(exc)[:200]
    return record


valid_paths = data.loc[data["resolved_path"].notna(), "resolved_path"].tolist()
log(f"reading metadata for {len(valid_paths):,} images, deep_check={CFG.deep_image_check}")

meta_records = [read_image_meta(p, CFG.deep_image_check) for p in tqdm(valid_paths, desc="images")]
img_meta = pd.DataFrame(meta_records, index=data.index[data["resolved_path"].notna()])

# A dataset that already ships a column called width, height, or mode would make the join
# raise. Rename any such source column rather than losing it or crashing.
clashes = [c for c in img_meta.columns if c in data.columns]
if clashes:
    print(f"Source columns renamed to avoid a clash with image metadata: {clashes}")
    renaming = {c: f"{c}_source" for c in clashes}
    data = data.rename(columns=renaming)
    CONDITION_COLUMNS = [renaming.get(c, c) for c in CONDITION_COLUMNS]
    ID_COLUMNS = [renaming.get(c, c) for c in ID_COLUMNS]
    PATH_COLUMNS = [renaming.get(c, c) for c in PATH_COLUMNS]
    META_COLUMNS = [renaming.get(c, c) for c in META_COLUMNS]
data = data.join(img_meta)

data["aspect_ratio"] = data["width"] / data["height"]
data["too_small"] = (data[["width", "height"]].min(axis=1) < CFG.min_image_side).fillna(False)
data["non_rgb"] = data["mode"].notna() & (data["mode"] != "RGB")

# A row whose image was never found is missing, not corrupted. Keeping the two counts apart
# matters because they have different causes and different fixes.
n_image_missing = int(data["resolved_path"].isna().sum())
n_corrupted = int((data["corrupted"].fillna(False) & data["resolved_path"].notna()).sum())
annotated_files = set(data.loc[data["resolved_path"].notna(), "resolved_path"])
n_orphan_files = len({str(p) for p in image_files} - annotated_files)

print(f"Rows with no image on disk : {n_image_missing:,}")
print(f"Found but unreadable       : {n_corrupted:,}")
print(f"Below {CFG.min_image_side} px on a side   : {int(data['too_small'].sum()):,}")
print(f"Not in RGB mode            : {int(data['non_rgb'].sum()):,}")
print(f"Image files with no annotation row : {n_orphan_files:,}")
print()
display(data[["width", "height", "aspect_ratio", "size_kb"]].describe().T)
print()
print("Color modes present:")
display(data["mode"].value_counts(dropna=False).to_frame("count"))

if int(data["non_rgb"].sum()) > 0:
    print(
        "\nNote: images outside RGB mode matter more than usual for this project. Redness and "
        "dark circles are color signals, so a grayscale or palette image carries strictly less "
        "information for those heads. These rows are flagged in the clean manifest."
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(data["width"], data["height"], s=6, alpha=0.3, edgecolors="none")
axes[0].set_xlabel("width in pixels")
axes[0].set_ylabel("height in pixels")
axes[0].set_title("Image resolution")

axes[1].hist(data["aspect_ratio"].dropna(), bins=60)
axes[1].axvline(1.0, color="black", linestyle=":", linewidth=1)
axes[1].set_xlabel("aspect ratio, width over height")
axes[1].set_ylabel("count")
axes[1].set_title("Aspect ratio distribution")

fig.suptitle("fig05 image resolution and aspect ratio", y=1.02)
save_fig(fig, "fig05_image_resolution")
plt.show()

AUDIT["image_stats"] = {
    "n_image_missing": n_image_missing,
    "n_corrupted": n_corrupted,
    "n_orphan_files": n_orphan_files,
    "n_too_small": int(data["too_small"].sum()),
    "n_non_rgb": int(data["non_rgb"].sum()),
    "width_median": float(data["width"].median()) if data["width"].notna().any() else None,
    "height_median": float(data["height"].median()) if data["height"].notna().any() else None,
    "modes": {str(k): int(v) for k, v in data["mode"].value_counts(dropna=False).items()},
}

## 5. Label distribution and imbalance

Three numbers decide whether a condition is usable: how often its label is missing, how the
mass is spread across severity levels, and how many samples the rarest usable levels have.
A condition whose labels are almost all at one level cannot teach a model anything, and a
condition with five samples at level 5 will produce a head that never predicts level 5.

In [ ]:
def normalized_entropy(counts: np.ndarray) -> float:
    """Compute Shannon entropy normalized to the range 0 to 1.

    A value of 1 means the mass is spread evenly across all possible levels. A value of 0
    means every sample sits at one level.

    Args:
        counts: Counts per level, including zeros for levels that never occur. The length of
            this array defines the number of possible levels.

    Returns:
        Normalized entropy between 0 and 1.
    """
    total = float(counts.sum())
    n_levels = len(counts)
    if total <= 0 or n_levels <= 1:
        return 0.0
    probabilities = counts[counts > 0] / total
    entropy = float(-(probabilities * np.log(probabilities)).sum())
    return entropy / math.log(n_levels)

In [ ]:
# Sanity checks for normalized_entropy, placed next to the function they verify.
assert abs(normalized_entropy(np.array([10, 10, 10, 10])) - 1.0) < 1e-9, "uniform must give 1.0"
assert normalized_entropy(np.array([40, 0, 0, 0])) == 0.0, "single level must give 0.0"
assert normalized_entropy(np.array([0, 0, 0, 0])) == 0.0, "empty must give 0.0"
assert 0.0 < normalized_entropy(np.array([90, 10, 0, 0])) < 1.0, "skewed must fall between"
print("normalized_entropy sanity checks passed")

In [ ]:
def condition_statistics(
    frame: pd.DataFrame, columns: list[str], levels: list[Any], config: AuditConfig
) -> pd.DataFrame:
    """Compute distribution statistics for every candidate condition.

    Args:
        frame: Annotation frame restricted to usable rows.
        columns: Candidate condition columns.
        levels: The global set of possible severity levels.
        config: Audit configuration holding the support threshold.

    Returns:
        One row per condition with missingness, entropy, and level support.
    """
    rows: list[dict[str, Any]] = []
    threshold = config.min_overall_samples_per_level
    for column in columns:
        series = frame[column]
        counts = np.array([int((series == level).sum()) for level in levels], dtype=float)
        n_valid = int(series.notna().sum())
        levels_with_support = int((counts >= threshold).sum())
        record: dict[str, Any] = {
            "condition": column,
            "n_valid": n_valid,
            "missing_rate": round(float(series.isna().mean()), 4),
            "normalized_entropy": round(normalized_entropy(counts), 4),
            "levels_present": int((counts > 0).sum()),
            "levels_with_support": levels_with_support,
            "majority_level": levels[int(np.argmax(counts))] if counts.sum() else None,
            "majority_share": round(float(counts.max() / counts.sum()), 4) if counts.sum() else None,
        }
        for level, count in zip(levels, counts):
            record[f"n_level_{level}"] = int(count)
        rows.append(record)
    return pd.DataFrame(rows)


if not CONDITION_COLUMNS:
    raise RuntimeError(
        "No candidate condition columns were detected, so there is nothing to analyse. Review "
        "the column profile in section 3 and correct CONDITION_COLUMNS manually before rerunning."
    )

usable = data.loc[~data["corrupted"].fillna(True)].copy()
print(f"Rows used for label statistics: {len(usable):,} of {len(data):,}")

cond_stats = condition_statistics(usable, CONDITION_COLUMNS, LEVELS, CFG)
cond_stats = cond_stats.sort_values("normalized_entropy", ascending=False).reset_index(drop=True)
display(cond_stats)
save_table(cond_stats, "phase1_condition_statistics")
AUDIT["condition_statistics"] = cond_stats.to_dict(orient="records")

In [ ]:
# A single condition order and color map, reused by every figure in every later notebook, so
# that a reader comparing two figures is never comparing two different color meanings.
CONDITION_ORDER: list[str] = cond_stats["condition"].tolist()
palette = sns.color_palette("husl", n_colors=len(CONDITION_ORDER))
CONDITION_COLORS: dict[str, str] = {
    condition: "#%02x%02x%02x" % tuple(int(255 * c) for c in rgb)
    for condition, rgb in zip(CONDITION_ORDER, palette)
}

palette_path = CFG.tables_dir / "condition_palette.json"
palette_path.write_text(
    json.dumps({"order": CONDITION_ORDER, "colors": CONDITION_COLORS}, indent=2), encoding="utf-8"
)
log(f"saved condition order and palette to {palette_path.name}")

In [ ]:
n_conditions = len(CONDITION_ORDER)
n_cols = 4
n_rows = math.ceil(n_conditions / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 2.8 * n_rows), sharex=True)
axes = np.atleast_1d(axes).ravel()

for axis, condition in zip(axes, CONDITION_ORDER):
    counts = [int((usable[condition] == level).sum()) for level in LEVELS]
    axis.bar([str(level) for level in LEVELS], counts, color=CONDITION_COLORS[condition])
    axis.axhline(CFG.min_overall_samples_per_level, color="black", linestyle=":", linewidth=1)
    entropy = float(cond_stats.loc[cond_stats["condition"] == condition, "normalized_entropy"].iloc[0])
    axis.set_title(f"{condition}\nentropy {entropy:.2f}", fontsize=9)
    axis.tick_params(labelsize=8)

for axis in axes[n_conditions:]:
    axis.set_visible(False)

fig.suptitle(
    f"fig01 severity level distribution per condition. Dotted line is the support threshold "
    f"of {CFG.min_overall_samples_per_level} samples per level.",
    y=1.01,
)
fig.tight_layout()
save_fig(fig, "fig01_level_distribution")
plt.show()

In [ ]:
fig, axis = plt.subplots(figsize=(10, max(3.5, 0.32 * len(CONDITION_ORDER))))
ordered = cond_stats.sort_values("missing_rate")
axis.barh(
    ordered["condition"],
    ordered["missing_rate"],
    color=[CONDITION_COLORS[c] for c in ordered["condition"]],
)
axis.axvline(CFG.max_missing_rate, color="black", linestyle="--", linewidth=1.2)
axis.set_xlabel("missing rate")
axis.set_title(
    f"fig02 missing rate per condition. Dashed line is the eligibility limit of "
    f"{CFG.max_missing_rate:.0%}."
)
fig.tight_layout()
save_fig(fig, "fig02_missing_rate")
plt.show()

## 6. Condition redundancy

Two conditions that always move together add a head and a loss term without adding
information. Spearman correlation is used rather than Pearson because the labels are ordinal,
so only the ranking is meaningful, not the spacing between levels.

In [ ]:
def spearman_matrix(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Compute pairwise Spearman correlation using pairwise complete observations.

    Args:
        frame: Annotation frame.
        columns: Condition columns.

    Returns:
        Square correlation dataframe indexed and columned by condition name.
    """
    size = len(columns)
    matrix = np.full((size, size), np.nan)
    for i, left in enumerate(columns):
        matrix[i, i] = 1.0
        # The diagonal is handled above and never selected as a pair, because selecting the
        # same column twice would build a two column frame with one repeated name, and
        # indexing that by name returns a frame rather than a series.
        for j in range(i + 1, size):
            right = columns[j]
            pair = frame[[left, right]].dropna()
            if len(pair) < 3 or pair[left].nunique() < 2 or pair[right].nunique() < 2:
                value = np.nan
            else:
                value = float(stats.spearmanr(pair[left], pair[right]).statistic)
            matrix[i, j] = value
            matrix[j, i] = value
    return pd.DataFrame(matrix, index=columns, columns=columns)


corr = spearman_matrix(usable, CONDITION_ORDER)

fig, axis = plt.subplots(figsize=(1.0 + 0.55 * len(CONDITION_ORDER), 0.9 + 0.5 * len(CONDITION_ORDER)))
sns.heatmap(
    corr, vmin=-1, vmax=1, cmap="RdBu_r", center=0, square=True,
    annot=len(CONDITION_ORDER) <= 12, fmt=".2f", annot_kws={"fontsize": 7},
    cbar_kws={"label": "Spearman rho"}, ax=axis,
)
axis.set_title("fig03 condition correlation, Spearman on pairwise complete rows")
fig.tight_layout()
save_fig(fig, "fig03_condition_correlation")
plt.show()

redundant_pairs = [
    {"condition_a": a, "condition_b": b, "spearman_rho": round(float(corr.loc[a, b]), 4)}
    for i, a in enumerate(CONDITION_ORDER)
    for b in CONDITION_ORDER[i + 1:]
    if pd.notna(corr.loc[a, b]) and abs(corr.loc[a, b]) > CFG.redundancy_spearman
]

print(f"Pairs above the redundancy threshold of {CFG.redundancy_spearman}: {len(redundant_pairs)}")
if redundant_pairs:
    display(pd.DataFrame(redundant_pairs))

AUDIT["redundant_pairs"] = redundant_pairs

## 7. Near duplicate detection

This is the single most important cell for the credibility of the final numbers. If the same
face appears in both the training split and the test split, the reported test score measures
memorisation rather than generalisation, and the whole evaluation becomes meaningless.

Perceptual hashing assigns a 64 bit fingerprint to each image. Images whose fingerprints differ
by at most a few bits are visually near identical. Every such cluster receives one group id,
and Phase 2 keeps a group inside a single split.

In [ ]:
POPCOUNT = np.array([bin(value).count("1") for value in range(256)], dtype=np.uint8)


class UnionFind:
    """Disjoint set structure used to merge near duplicate images into groups."""

    def __init__(self, size: int) -> None:
        """Initialise every element as its own group.

        Args:
            size: Number of elements.
        """
        self.parent = list(range(size))

    def find(self, item: int) -> int:
        """Return the representative of an element, compressing the path on the way.

        Args:
            item: Element index.

        Returns:
            Index of the group representative.
        """
        root = item
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[item] != root:
            self.parent[item], item = root, self.parent[item]
        return root

    def union(self, left: int, right: int) -> None:
        """Merge the groups containing two elements.

        Args:
            left: First element index.
            right: Second element index.
        """
        left_root, right_root = self.find(left), self.find(right)
        if left_root != right_root:
            self.parent[right_root] = left_root


def hamming_pairs(packed: np.ndarray, max_distance: int, memory_budget_mb: int = 48) -> np.ndarray:
    """Find all index pairs whose packed hashes are within a Hamming distance.

    The comparison is done in chunks with a popcount lookup table, which keeps an all pairs
    comparison tractable without pulling in an extra dependency.

    Args:
        packed: Array of shape (n, n_bytes) holding one packed hash per row.
        max_distance: Inclusive Hamming distance threshold.
        memory_budget_mb: Approximate ceiling for the temporary XOR block.

    Returns:
        Array of shape (n_pairs, 2) holding index pairs with i less than j.
    """
    n_items, n_bytes = packed.shape
    if n_items < 2:
        return np.empty((0, 2), dtype=np.int64)

    rows_per_chunk = max(16, int((memory_budget_mb * 1e6) / max(n_items * n_bytes, 1)))
    collected: list[np.ndarray] = []
    for start in range(0, n_items, rows_per_chunk):
        stop = min(start + rows_per_chunk, n_items)
        block = packed[start:stop]
        distances = POPCOUNT[np.bitwise_xor(block[:, None, :], packed[None, :, :])].sum(axis=2)
        local_i, global_j = np.nonzero(distances <= max_distance)
        global_i = local_i + start
        keep = global_i < global_j
        if keep.any():
            collected.append(np.stack([global_i[keep], global_j[keep]], axis=1))
    return np.concatenate(collected) if collected else np.empty((0, 2), dtype=np.int64)


def group_by_hash(packed: np.ndarray, max_distance: int) -> np.ndarray:
    """Assign a group id to each item so that near duplicates share an id.

    Args:
        packed: Array of shape (n, n_bytes) holding one packed hash per row.
        max_distance: Inclusive Hamming distance threshold.

    Returns:
        Array of group ids with one entry per item.
    """
    pairs = hamming_pairs(packed, max_distance)
    union = UnionFind(packed.shape[0])
    for left, right in pairs:
        union.union(int(left), int(right))
    roots = np.array([union.find(i) for i in range(packed.shape[0])])
    _unique, group_ids = np.unique(roots, return_inverse=True)
    return group_ids

In [ ]:
# Sanity checks for the grouping logic, using hand built hashes with known distances.
_probe = np.array(
    [
        [0b00000000, 0, 0, 0, 0, 0, 0, 0],  # item 0
        [0b00000001, 0, 0, 0, 0, 0, 0, 0],  # item 1, distance 1 from item 0
        [0b11111111, 255, 255, 255, 255, 255, 255, 255],  # item 2, far from everything
        [0b00000011, 0, 0, 0, 0, 0, 0, 0],  # item 3, distance 2 from item 0
    ],
    dtype=np.uint8,
)
_groups = group_by_hash(_probe, max_distance=2)
assert _groups[0] == _groups[1] == _groups[3], "items within distance 2 must share a group"
assert _groups[2] != _groups[0], "a distant item must form its own group"

_strict = group_by_hash(_probe, max_distance=0)
assert len({int(g) for g in _strict}) == 4, "distance 0 must keep all four items separate"
print("near duplicate grouping sanity checks passed")

In [ ]:
hashable = usable.loc[usable["resolved_path"].notna()].copy()
paths = hashable["resolved_path"].tolist()
log(f"computing perceptual hashes for {len(paths):,} images")

hash_values: list[str | None] = []
for path in tqdm(paths, desc="phash"):
    try:
        with Image.open(path) as image:
            hash_values.append(str(imagehash.phash(image.convert("RGB"), hash_size=CFG.phash_size)))
    except Exception:  # noqa: BLE001 - unreadable images are already flagged elsewhere
        hash_values.append(None)

hashable["phash"] = hash_values
with_hash = hashable.loc[hashable["phash"].notna()].copy()
log(f"hashed {len(with_hash):,} images successfully")

packed = np.array(
    [np.packbits([int(bit) for bit in bin(int(h, 16))[2:].zfill(CFG.phash_size ** 2)]) for h in with_hash["phash"]],
    dtype=np.uint8,
)
print("packed hash matrix shape:", packed.shape)

start_time = time.time()
group_ids = group_by_hash(packed, CFG.phash_max_distance)
log(f"grouping finished in {time.time() - start_time:.1f} s")

with_hash["group_id"] = group_ids
group_sizes = pd.Series(group_ids).value_counts()

print(f"\nImages hashed          : {len(with_hash):,}")
print(f"Distinct groups        : {group_sizes.size:,}")
print(f"Groups with duplicates : {int((group_sizes > 1).sum()):,}")
print(f"Images inside a duplicate group : {int(group_sizes[group_sizes > 1].sum()):,}")
print(f"Largest group size     : {int(group_sizes.max())}")
print()
display(group_sizes.value_counts().sort_index().rename_axis("group_size").to_frame("n_groups").head(15))

exact_duplicates = int(with_hash["phash"].duplicated().sum())
print(f"\nImages with a byte identical perceptual hash: {exact_duplicates:,}")

AUDIT["duplicates"] = {
    "phash_max_distance": CFG.phash_max_distance,
    "n_hashed": int(len(with_hash)),
    "n_groups": int(group_sizes.size),
    "n_groups_with_duplicates": int((group_sizes > 1).sum()),
    "n_images_in_duplicate_groups": int(group_sizes[group_sizes > 1].sum()),
    "largest_group": int(group_sizes.max()),
    "n_exact_hash_duplicates": exact_duplicates,
}

In [ ]:
duplicate_groups = with_hash[["group_id"]].copy()
duplicate_groups["image_path"] = with_hash["resolved_path"].apply(lambda p: str(Path(p).relative_to(DATASET_ROOT)))
duplicate_groups["phash"] = with_hash["phash"]
duplicate_groups = duplicate_groups[["image_path", "group_id", "phash"]].sort_values("group_id")
save_table(duplicate_groups, "phase1_duplicate_groups")
print("This table is the handoff to Phase 2. The split must never place one group id in two splits.")

In [ ]:
if CFG.show_sample_images:
    multi = group_sizes[group_sizes > 1].index.tolist()[:4]
    if multi:
        n_show = min(4, len(multi))
        fig, axes = plt.subplots(n_show, 3, figsize=(9, 3 * n_show))
        axes = np.atleast_2d(axes)
        for row, group in enumerate(multi[:n_show]):
            members = with_hash.loc[with_hash["group_id"] == group, "resolved_path"].tolist()[:3]
            for col in range(3):
                axis = axes[row, col]
                axis.axis("off")
                if col < len(members):
                    with Image.open(members[col]) as image:
                        axis.imshow(image.convert("RGB"))
                    axis.set_title(f"group {group}", fontsize=8)
        fig.suptitle("Internal review: examples of near duplicate clusters", y=1.0)
        fig.tight_layout()
        save_fig(fig, "internal_duplicate_pairs")
        plt.show()
    else:
        print("No duplicate clusters were found, so there is nothing to display.")
else:
    print("Image rendering is disabled through the configuration.")

## 8. Face framing check

Whether the images are already tight portraits or wide shots decides whether Phase 2 needs a
face detection and cropping step. A wide shot wastes most of the input resolution on
background, which directly hurts small signals such as pore texture. This cell only measures
and recommends. It does not implement cropping, because that decision belongs to Gate 1.

In [ ]:
def measure_face_framing(paths: list[str], max_width: int = 640) -> pd.DataFrame:
    """Measure how much of each image is occupied by a detected frontal face.

    Args:
        paths: Image paths to sample.
        max_width: Images wider than this are downscaled before detection, for speed.

    Returns:
        One row per image with detection count, face area ratio, and center offset.
    """
    cascade_file = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(str(cascade_file))
    rows: list[dict[str, Any]] = []

    for path in tqdm(paths, desc="face check"):
        image = cv2.imread(path)
        if image is None:
            continue
        height, width = image.shape[:2]
        scale = min(1.0, max_width / width)
        resized = cv2.resize(image, (int(width * scale), int(height * scale))) if scale < 1.0 else image
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
        faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))

        record: dict[str, Any] = {"path": path, "n_faces": len(faces), "face_area_ratio": None, "center_offset": None}
        if len(faces):
            fx, fy, fw, fh = max(faces, key=lambda box: box[2] * box[3])
            frame_h, frame_w = gray.shape[:2]
            record["face_area_ratio"] = float((fw * fh) / (frame_w * frame_h))
            face_cx, face_cy = fx + fw / 2, fy + fh / 2
            offset_x = abs(face_cx - frame_w / 2) / frame_w
            offset_y = abs(face_cy - frame_h / 2) / frame_h
            record["center_offset"] = float(math.hypot(offset_x, offset_y))
        rows.append(record)
    return pd.DataFrame(rows)


sample_paths = (
    with_hash["resolved_path"]
    .sample(n=min(CFG.face_sample_size, len(with_hash)), random_state=CFG.seed)
    .tolist()
)
framing = measure_face_framing(sample_paths)

detection_rate = float((framing["n_faces"] > 0).mean()) if len(framing) else 0.0
median_area = float(framing["face_area_ratio"].median()) if framing["face_area_ratio"].notna().any() else float("nan")
median_offset = float(framing["center_offset"].median()) if framing["center_offset"].notna().any() else float("nan")

print(f"Sample size            : {len(framing)}")
print(f"Face detection rate    : {detection_rate:.1%}")
print(f"Median face area ratio : {median_area:.3f}")
print(f"Median center offset   : {median_offset:.3f}")

if detection_rate < 0.5:
    recommendation = (
        "A frontal face detector found a face in fewer than half the sampled images. Either the "
        "images are already tight crops of skin regions rather than whole faces, or they are not "
        "frontal. Inspect the sample grid below before deciding. Do not add a detector blindly."
    )
elif median_area >= 0.35:
    recommendation = (
        "Faces already fill a large share of the frame, so a cropping step would add complexity "
        "and a new failure mode for little gain. Recommendation: train on the full image with a "
        "resize and a center crop."
    )
else:
    recommendation = (
        "Faces occupy a small share of the frame, so most of the input resolution is spent on "
        "background. Recommendation: add a face detection and crop step in Phase 2, cache the "
        "crops, and apply the identical crop at inference time in the app."
    )

print("\nRecommendation for Gate 1:")
print(recommendation)

AUDIT["face_framing"] = {
    "sample_size": int(len(framing)),
    "detection_rate": round(detection_rate, 4),
    "median_face_area_ratio": None if math.isnan(median_area) else round(median_area, 4),
    "median_center_offset": None if math.isnan(median_offset) else round(median_offset, 4),
    "recommendation": recommendation,
}

In [ ]:
if CFG.show_sample_images:
    grid_paths = sample_paths[:12]
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for axis, path in zip(axes.ravel(), grid_paths):
        with Image.open(path) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(Path(path).name[:24], fontsize=7)
        axis.axis("off")
    for axis in axes.ravel()[len(grid_paths):]:
        axis.axis("off")
    fig.suptitle("Internal review: random image sample used for the framing check", y=1.0)
    fig.tight_layout()
    save_fig(fig, "internal_framing_sample")
    plt.show()

In [ ]:
if CFG.show_sample_images:
    top_conditions = CONDITION_ORDER[:2]
    for condition in top_conditions:
        present_levels = [lv for lv in LEVELS if int((with_hash[condition] == lv).sum()) > 0]
        if not present_levels:
            continue
        fig, axes = plt.subplots(len(present_levels), 3, figsize=(9, 2.9 * len(present_levels)))
        axes = np.atleast_2d(axes)
        for row, level in enumerate(present_levels):
            subset = with_hash.loc[with_hash[condition] == level, "resolved_path"]
            picks = subset.sample(n=min(3, len(subset)), random_state=CFG.seed).tolist()
            for col in range(3):
                axis = axes[row, col]
                axis.axis("off")
                if col < len(picks):
                    with Image.open(picks[col]) as image:
                        axis.imshow(image.convert("RGB"))
                    if col == 0:
                        axis.set_title(f"{condition} level {level}", fontsize=9, loc="left")
        fig.suptitle(f"fig04 sample images by severity level, condition: {condition}", y=1.0)
        fig.tight_layout()
        save_fig(fig, f"fig04_samples_{condition}")
        plt.show()

    print(
        "These grids are for internal review only. Dataset images are not redistributed, so do "
        "not commit them to the repository."
    )

## 9. Cleaning decisions and the clean manifest

Cleaning here means recording decisions, not quietly deleting data. Every row keeps a reason
for why it is kept or dropped, so the decision is auditable and reversible.

One rule is deliberate and worth stating plainly: a missing label is never filled in. A guessed
label would look like supervision while teaching the model something untrue. Missing labels are
masked out of the loss instead, which is what the mask field in the dataset is for.

In [ ]:
rules: dict[str, pd.Series] = {
    "image_missing": data["resolved_path"].isna(),
    "image_corrupted": data["corrupted"].fillna(True) & data["resolved_path"].notna(),
    "image_too_small": data["too_small"].fillna(False),
    "image_non_rgb": data["non_rgb"].fillna(False),
    "all_labels_missing": data[CONDITION_COLUMNS].isna().all(axis=1),
}

# The range test compares against the range the proposal assumes, not against the range
# observed in the data. Comparing against the observed range would make the test vacuous,
# since the observed minimum and maximum can never fall outside themselves.
out_of_range = pd.Series(False, index=data.index)
for column in CONDITION_COLUMNS:
    values = pd.to_numeric(data[column], errors="coerce")
    out_of_range |= values.notna() & (
        (values < CFG.expected_level_min) | (values > CFG.expected_level_max)
    )
rules["label_outside_expected_range"] = out_of_range

# Blocking rules make a row unusable. Advisory rules are recorded but do not remove the row,
# because losing data is a real cost and these conditions are recoverable or tolerable.
BLOCKING = ["image_missing", "image_corrupted", "all_labels_missing"]
ADVISORY = [name for name in rules if name not in BLOCKING]

summary = pd.DataFrame(
    {
        "rule": list(rules),
        "n_rows": [int(mask.sum()) for mask in rules.values()],
        "share": [round(float(mask.mean()), 4) for mask in rules.values()],
        "action": ["drop" if name in BLOCKING else "flag only" for name in rules],
    }
)
display(summary)
save_table(summary, "phase1_cleaning_summary")

def join_active_rules(names: list[str], rule_map: dict[str, pd.Series], n_rows: int) -> list[str]:
    """Build a comma separated reason string per row from a set of boolean rules.

    A vectorised matrix is used instead of per row indexing, because a per row pandas lookup
    would turn a fast operation into minutes of work on a large dataset.

    Args:
        names: Rule names to include.
        rule_map: Mapping from rule name to a boolean mask over the rows.
        n_rows: Number of rows.

    Returns:
        One string per row, empty when no rule in the set fired.
    """
    if not names:
        return [""] * n_rows
    matrix = np.column_stack([rule_map[name].to_numpy(dtype=bool) for name in names])
    return [", ".join(np.array(names)[row]) for row in matrix]


blocked = pd.concat([rules[name] for name in BLOCKING], axis=1).any(axis=1)
data["keep"] = ~blocked
data["drop_reason"] = join_active_rules(BLOCKING, rules, len(data))
data["quality_flags"] = join_active_rules(ADVISORY, rules, len(data))

print(f"\nRows kept    : {int(data['keep'].sum()):,}")
print(f"Rows dropped : {int((~data['keep']).sum()):,}")
AUDIT["cleaning"] = summary.to_dict(orient="records")
AUDIT["n_rows_kept"] = int(data["keep"].sum())

In [ ]:
group_lookup = dict(zip(with_hash["resolved_path"], with_hash["group_id"]))
data["group_id"] = pd.to_numeric(data["resolved_path"].map(group_lookup), errors="coerce")

# Any kept row without a perceptual hash gets a unique group id of its own, so that the Phase 2
# grouped split has a complete grouping and never silently treats a missing id as one big group.
missing_group = data["keep"] & data["group_id"].isna()
if missing_group.any():
    next_id = int(data["group_id"].max()) + 1 if data["group_id"].notna().any() else 0
    data.loc[missing_group, "group_id"] = list(range(next_id, next_id + int(missing_group.sum())))
    print(f"Assigned singleton group ids to {int(missing_group.sum())} rows without a hash.")

data["group_id"] = data["group_id"].astype("Int64")

n_groups_kept = int(data.loc[data["keep"], "group_id"].nunique())
print(f"Distinct group ids among kept rows: {n_groups_kept:,}")
print(
    "Phase 2 splits on this column. The effective sample size for the split is the number of "
    "groups, not the number of rows, so a large gap between the two means less independent data "
    "than the row count suggests."
)

manifest_columns = (
    ["resolved_path", "group_id", "keep", "drop_reason", "quality_flags", "width", "height", "mode"]
    + CONDITION_COLUMNS
    + [c for c in ID_COLUMNS + PATH_COLUMNS + META_COLUMNS if c in data.columns]
)
manifest = data[manifest_columns].copy()
manifest["image_path"] = manifest["resolved_path"].apply(
    lambda p: str(Path(p).relative_to(DATASET_ROOT)) if isinstance(p, str) else None
)
manifest = manifest.drop(columns=["resolved_path"])

save_table(manifest, "phase1_clean_manifest")
print("\nClean manifest preview:")
display(manifest.head(8))

assert manifest.loc[manifest["keep"], "group_id"].notna().all(), "every kept row needs a group id"
assert not manifest.loc[manifest["keep"], "image_path"].isna().any(), "every kept row needs an image"
print("clean manifest sanity checks passed")

## 10. Condition selection

The selection rule comes from the project requirements document, section 7.1. A condition is
eligible when all three criteria hold:

1. Its missing rate is below the configured limit.
2. At least three severity levels each have enough samples to learn from.
3. Its normalized label entropy is at or above the configured floor.

Eligible conditions are then ranked by entropy, redundant partners are dropped, and acne is
forced in because the baseline model and the external ACNE04 evaluation both depend on it.

One honest caveat is printed with the result. The support criterion is written in terms of the
training split, which does not exist yet, so this notebook converts it into a whole dataset
requirement using the planned training fraction. Phase 2 recomputes it on the real split.

In [ ]:
def evaluate_eligibility(stats_frame: pd.DataFrame, config: AuditConfig) -> pd.DataFrame:
    """Apply the three eligibility criteria to every candidate condition.

    Args:
        stats_frame: Output of condition_statistics.
        config: Audit configuration holding the thresholds.

    Returns:
        The input frame with one boolean column per criterion, an overall eligible flag, and a
        human readable reason for any rejection.
    """
    result = stats_frame.copy()
    result["passes_missingness"] = result["missing_rate"] < config.max_missing_rate
    result["passes_support"] = result["levels_with_support"] >= config.min_levels_with_support
    result["passes_entropy"] = result["normalized_entropy"] >= config.min_normalized_entropy
    result["eligible"] = result["passes_missingness"] & result["passes_support"] & result["passes_entropy"]

    reasons: list[str] = []
    for _, row in result.iterrows():
        failed: list[str] = []
        if not row["passes_missingness"]:
            failed.append(f"missing rate {row['missing_rate']:.2f} is not below {config.max_missing_rate}")
        if not row["passes_support"]:
            failed.append(
                f"only {row['levels_with_support']} level(s) reach "
                f"{config.min_overall_samples_per_level} samples, need {config.min_levels_with_support}"
            )
        if not row["passes_entropy"]:
            failed.append(
                f"entropy {row['normalized_entropy']:.2f} is below {config.min_normalized_entropy}"
            )
        reasons.append("; ".join(failed) or "eligible")
    result["reason"] = reasons
    return result


def select_conditions(
    eligibility: pd.DataFrame, correlation: pd.DataFrame, config: AuditConfig
) -> tuple[list[str], list[dict[str, Any]]]:
    """Rank eligible conditions, drop redundant partners, and force required conditions in.

    Args:
        eligibility: Output of evaluate_eligibility.
        correlation: Square Spearman correlation frame.
        config: Audit configuration.

    Returns:
        The proposed condition list and a decision log explaining every inclusion or exclusion.
    """
    ranked = eligibility.loc[eligibility["eligible"]].sort_values("normalized_entropy", ascending=False)
    decisions: list[dict[str, Any]] = []
    chosen: list[str] = []

    # A forced token such as "acne" can match several columns, for example "acne" and
    # "acne_scars". Prefer an exact name match so the forced slot is not silently spent twice.
    forced: list[str] = []
    for token in config.forced_conditions:
        matches = [c for c in eligibility["condition"] if token in str(c).lower()]
        exact = [c for c in matches if str(c).lower() == token]
        if len(matches) > 1:
            print(
                f"Note: the forced token '{token}' matches {matches}. "
                f"Using {exact or matches[:1]}. Confirm this at Gate 1."
            )
        forced.extend(exact[:1] or matches[:1])

    for condition in forced:
        is_eligible = bool(eligibility.loc[eligibility["condition"] == condition, "eligible"].iloc[0])
        chosen.append(condition)
        decisions.append(
            {
                "condition": condition,
                "decision": "included, forced by the requirements document",
                "note": "meets every criterion" if is_eligible else "FORCED IN DESPITE FAILING A CRITERION",
            }
        )

    for condition in ranked["condition"]:
        if condition in chosen:
            continue
        if len(chosen) >= config.target_condition_count:
            decisions.append({"condition": condition, "decision": "not included", "note": "target count already reached"})
            continue
        partner = next(
            (
                other for other in chosen
                if other in correlation.columns
                and condition in correlation.index
                and pd.notna(correlation.loc[condition, other])
                and abs(correlation.loc[condition, other]) > config.redundancy_spearman
            ),
            None,
        )
        if partner is not None:
            decisions.append(
                {
                    "condition": condition,
                    "decision": "dropped as redundant",
                    "note": f"Spearman {correlation.loc[condition, partner]:.2f} with {partner}",
                }
            )
            continue
        chosen.append(condition)
        decisions.append({"condition": condition, "decision": "included", "note": "highest remaining entropy"})

    for condition in eligibility.loc[~eligibility["eligible"], "condition"]:
        if condition not in chosen:
            reason = eligibility.loc[eligibility["condition"] == condition, "reason"].iloc[0]
            decisions.append({"condition": condition, "decision": "not eligible", "note": reason})

    return chosen, decisions

In [ ]:
# Sanity checks for the selection rule, using a synthetic table where the answer is known.
_probe_stats = pd.DataFrame(
    {
        "condition": ["good", "too_missing", "too_flat", "too_few_levels"],
        "missing_rate": [0.05, 0.50, 0.05, 0.05],
        "normalized_entropy": [0.90, 0.90, 0.10, 0.90],
        "levels_with_support": [5, 5, 5, 1],
    }
)
_probe_eligibility = evaluate_eligibility(_probe_stats, CFG)
_verdict = dict(zip(_probe_eligibility["condition"], _probe_eligibility["eligible"]))
assert _verdict["good"], "a condition meeting every criterion must be eligible"
assert not _verdict["too_missing"], "a high missing rate must disqualify"
assert not _verdict["too_flat"], "low entropy must disqualify"
assert not _verdict["too_few_levels"], "too few supported levels must disqualify"
print("eligibility rule sanity checks passed")

In [ ]:
eligibility = evaluate_eligibility(cond_stats, CFG)
display(
    eligibility[
        [
            "condition", "missing_rate", "normalized_entropy", "levels_with_support",
            "passes_missingness", "passes_entropy", "passes_support", "eligible", "reason",
        ]
    ]
)
save_table(eligibility, "phase1_condition_eligibility")

proposed, decision_log = select_conditions(eligibility, corr, CFG)
decisions_frame = pd.DataFrame(decision_log)
display(decisions_frame)
save_table(decisions_frame, "phase1_selection_decisions")

n_eligible = int(eligibility["eligible"].sum())
print(f"\nEligible conditions: {n_eligible} of {len(eligibility)}")
print(f"Target count from the requirements document: {CFG.target_condition_count}")
print(f"Proposed set ({len(proposed)}):")
for condition in proposed:
    print("  ", condition)

if len(proposed) < CFG.target_condition_count:
    print(
        f"\nATTENTION: only {len(proposed)} conditions qualify, fewer than the {CFG.target_condition_count} "
        f"the proposal plans for. Do not pad the list with conditions that failed a criterion. The "
        f"options are to model fewer conditions, or to relax a threshold deliberately and record "
        f"that decision. This belongs to Gate 1."
    )

print("\nPaste this into the config cell of every later notebook once Gate 1 approves it:")
print(f"SELECTED_CONDITIONS = {proposed}")

AUDIT["selection"] = {
    "n_eligible": n_eligible,
    "proposed": proposed,
    "decisions": decision_log,
    "support_caveat": (
        f"The per level support threshold of {CFG.min_train_samples_per_level} training samples was "
        f"converted to {CFG.min_overall_samples_per_level} whole dataset samples using an assumed "
        f"training fraction of {CFG.assumed_train_fraction}. Phase 2 must recheck this on the real split."
    ),
}

## 11. Audit report and Gate 1 summary

Everything measured above is written to a single JSON file so the findings can be quoted in the
report without rerunning the notebook.

In [ ]:
AUDIT["generated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
AUDIT["condition_columns"] = CONDITION_COLUMNS

report_path = CFG.working_root / "audit_report.json"
report_path.write_text(json.dumps(AUDIT, indent=2, default=str), encoding="utf-8")
log(f"wrote {report_path} with {len(json.dumps(AUDIT, default=str)) / 1024:.1f} KB")

print("\nFiles produced in /kaggle/working:")
for path in sorted(CFG.working_root.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(CFG.working_root)}  ({path.stat().st_size / 1024:.1f} KB)")

In [ ]:
separator = "=" * 78
print(separator)
print("GATE 1 SUMMARY. Report these four answers before Phase 2 begins.")
print(separator)

print("\n(a) Label schema findings")
print(f"    Annotation table     : {AUDIT.get('primary_table')}")
print(f"    Rows                 : {AUDIT['n_rows']:,}, with a resolvable image: {AUDIT['n_rows_with_image']:,}")
print(f"    Condition columns    : {len(CONDITION_COLUMNS)} (proposal assumed {CFG.expected_condition_count})")
print(f"    Severity levels      : {LEVELS} (proposal assumed {CFG.expected_level_min} to {CFG.expected_level_max})")
print(f"    Assumption mismatches: {AUDIT['n_schema_mismatches']}")
for check in AUDIT["schema_checks"]:
    if check["status"] != "PASS":
        print(f"      {check['status']}: {check['assumption']} -> {check['observed']}")

print("\n(b) Proposed conditions")
print(f"    {proposed}")
print(f"    Eligible: {n_eligible} of {len(eligibility)}. Target: {CFG.target_condition_count}.")

print("\n(c) Face cropping recommendation")
print(f"    Detection rate {AUDIT['face_framing']['detection_rate']:.1%}, "
      f"median face area ratio {AUDIT['face_framing']['median_face_area_ratio']}")
print(f"    {AUDIT['face_framing']['recommendation']}")

print("\n(d) Data quality and leakage risk")
print(f"    Rows kept after cleaning     : {AUDIT['n_rows_kept']:,}")
print(f"    Rows with no image on disk   : {AUDIT['image_stats']['n_image_missing']:,}")
print(f"    Found but unreadable         : {AUDIT['image_stats']['n_corrupted']:,}")
print(f"    Image files never annotated  : {AUDIT['image_stats']['n_orphan_files']:,}")
print(f"    Images not in RGB mode       : {AUDIT['image_stats']['n_non_rgb']:,}")
print(f"    Near duplicate groups        : {AUDIT['duplicates']['n_groups_with_duplicates']:,}")
print(f"    Images inside those groups   : {AUDIT['duplicates']['n_images_in_duplicate_groups']:,}")
print(f"    Largest duplicate group      : {AUDIT['duplicates']['largest_group']}")

print("\n" + separator)
print("STOP HERE. Phase 2 needs human approval of the points above.")
print(separator)

## 12. What these findings mean for the model

The audit is not an end in itself. Each measurement above maps to a specific decision that
protects the model from underfitting or overfitting later.

1. **Near duplicate groups drive the split.** Phase 2 splits by group id, never by row. Without
   this, a duplicated face lands in both training and test, the test score rises, and the rise
   is pure memorisation. This is the difference between a result that survives scrutiny and one
   that does not.
2. **Missing rates decide the masking, not imputation.** Conditions are scored independently, so
   a row can supervise one head and stay silent for another. The mask field carries that, and a
   head with no valid labels in a batch contributes exactly zero to the loss rather than a NaN.
3. **Level imbalance decides the loss.** Where the mass concentrates on one or two levels, plain
   cross entropy collapses onto the majority level and reports a respectable accuracy while
   being useless. That is why the plan uses class weighted cross entropy and quadratic weighted
   kappa as the selection metric, and why macro F1 is monitored per head.
4. **Sparse high severity levels bound the ambition.** A level with very few samples will not be
   learned reliably no matter which backbone is used. The honest responses are to report per
   level recall openly, or to merge the sparsest levels, and to decide which at Gate 1 rather
   than after seeing test numbers.
5. **Entropy and redundancy decide the head count.** Two conditions correlated above the
   threshold do not justify two heads. Fewer, better separated heads mean fewer parameters
   spent on duplicated signal and less room to overfit.
6. **Color mode matters more here than in a typical vision task.** Redness and dark circles are
   color signals. Any image stored outside RGB carries less of exactly the signal two heads
   need, which is also why the augmentation policy forbids hue and saturation jitter.
7. **Image resolution informs the input size.** If most images are far larger than the planned
   224 pixel input, then fine texture is being discarded at resize time, and the resolution
   ablation planned for Phase 6 is worth the compute.

None of these choices are finalised here. They are the agenda for Gate 1.

## Next step

1. Download everything in `/kaggle/working/`.
2. Commit `audit_report.json`, the tables, and the numbered figures into the repository. Do not
   commit the internal image grids, since the dataset is not redistributable.
3. Report the Gate 1 summary and wait for approval before starting Phase 2.